# Fine-tuning T5 Model with JSON Data (Final Version)


**Step 1: Install Required Libraries**

In [6]:
!pip install transformers datasets accelerate evaluate sentencepiece

**Step 2: Import Libraries**

In [7]:
import os
import json
from datasets import Dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, TrainingArguments, Trainer

In [38]:
import os
import json
import torch
import numpy as np
import evaluate

from datasets import Dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Trainer,
    TrainingArguments,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

**Step 3: Load Data**

Update `DATA_FILE_PATH` to your JSON file path.

In [40]:
DATA_FILE_PATH = '/kaggle/input/datasets/jehad24/response-generator/training_data_with_responses (1).json'

def load_json_data(file_path):
    if not os.path.exists(file_path):
        print(f"File {file_path} not found. Creating a sample for demonstration.")
        sample_data = {
            "TRAINING_DATA": [
                ["Do you remember my previous order?", "memory_check", "Yes! I remember."],
                ["بعد إذنك، سلام يا فرح", "goodbye", "مع السلامة!"]
            ]
        }
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(sample_data, f, ensure_ascii=False, indent=2)
            
    with open(file_path, 'r', encoding='utf-8') as f:
        full_data = json.load(f)
            
    processed_data = []
    for item in full_data['TRAINING_DATA']:
        processed_data.append({'input_text': item[0], 'target_text': item[2]})
            
    return Dataset.from_list(processed_data)

full_dataset = load_json_data(DATA_FILE_PATH)
print(f"Loaded {len(full_dataset)} examples.")


Loaded 54000 examples.


# CELL 5 — Train/Test Split

In [ ]:
split_dataset = full_dataset.train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"Train size: {len(train_dataset)}")
print(f"Eval size: {len(eval_dataset)}")

**Step 4: Tokenization**

In [41]:
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name, legacy=False)

def preprocess_function(examples):

    model_inputs = tokenizer(
        examples["input_text"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=examples["target_text"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_eval = eval_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/43200 [00:00<?, ? examples/s]

Map:   0%|          | 0/10800 [00:00<?, ? examples/s]

**Step 5: Training**

In [44]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 97.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 89.8 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [47]:
model = T5ForConditionalGeneration.from_pretrained(model_name)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    logging_steps=50,
    save_strategy="epoch",
    eval_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available()
)

model.to("cuda" if torch.cuda.is_available() else "cpu")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train
)

print("Starting training...")
trainer.train()
print("Training finished.")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Starting training...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
50,23.279490
100,2.951614
150,1.636300
200,1.230549
250,0.909738
300,0.736595
350,0.639456
400,0.593937
450,0.541341
500,0.492101


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training finished.


**Step 6: Save and Test**

In [50]:
def generate_response(text):

    model.eval()

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=128,
            num_beams=5,
            early_stopping=True,
            no_repeat_ngram_size=2
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

sample_input = "بعد إذنك، سلام "

print("User:", sample_input)
print("Bot:", generate_response(sample_input))

User: بعد إذنك، سلام 
Bot: ! Assistify AI® progresses, we can assist you in the delivery of products. What can I help you with?


In [51]:
sample_input = "hello "

print("User:", sample_input)
print("Bot:", generate_response(sample_input))

User: hello 
Bot: Hello! I'm Assistify AI, your medical store assistant. How can I help you today?


In [48]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    return {
        "rouge1": result["rouge1"],
        "rouge2": result["rouge2"],
        "rougeL": result["rougeL"]
    }

eval_trainer = Seq2SeqTrainer(
    model=model,
    args=Seq2SeqTrainingArguments(
        output_dir="./results",
        per_device_eval_batch_size=8,
        predict_with_generate=True,
        report_to="none"
    ),
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics
)

print("Evaluating...")
metrics = eval_trainer.evaluate()

print("\n--- Results ---")
for k, v in metrics.items():
    print(k, ":", v)

Evaluating...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



--- Results ---
eval_loss : 0.029832368716597557
eval_rouge1 : 0.3478399994483544
eval_rouge2 : 0.3139753313514345
eval_rougeL : 0.34295942351258885
eval_runtime : 210.3295
eval_samples_per_second : 51.348
eval_steps_per_second : 3.209
